# Applied Deep Learning Course - Introduction

This is the introduction to the Deep Learning course.

This component will cover

 * Course Background
 * Course Scope
 * Prerequisites Information
 * Course Structure
 * Assignments and Assessment
 * Coding
 * A little bit of background on Linear Algebra

## Coarse Scope

Let's get to the point. Modern frameworks make it easy to code up some solutions for Deep Learning. Have a look....

### Tensorflow: Accurate Digit Classification in 14 Lines

In [2]:
import sys
import tensorflow as tf
print(sys.version)
print(tf.__version__)

3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
2.19.0


In [3]:
# Get a copy of the mnist dataset container
mnist = tf.keras.datasets.mnist

# Pull out the training and test data
(x_train, y_train),(x_test, y_test) = mnist.load_data()

# Normalize the training and test datasets
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

# Create a simple sequential network object
model = tf.keras.models.Sequential()

# Add layers to the network for processing the input data
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(64, activation=tf.keras.activations.sigmoid))
model.add(tf.keras.layers.Dense(128, activation=tf.keras.activations.sigmoid))
model.add(tf.keras.layers.Dense(10, activation=tf.keras.activations.softmax))

# Compile the model
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Start the training process
model.fit(x=x_train, y=y_train, epochs=5)

print(model.summary())

# Evaluate the model performance with test data
test_loss, test_acc = model.evaluate(x=x_test, y=y_test,verbose=0)

# Print out the model accuracy
print('\nTest accuracy: ' + str(test_acc*100) + "%" )

# Save the model
model.save("my_model_file.keras")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.6902 - loss: 1.0928
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9237 - loss: 0.2593
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9439 - loss: 0.1892
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9556 - loss: 0.1516
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9622 - loss: 0.1268


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (32, 784)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 64)               │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 128)              │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (32, 10)               │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 179,552 (701.38 KB)

 Trainable params: 59,850 (233.79 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 119,702 (467.59 KB)

None

Test accuracy: 96.07999920845032%


**Aside on model saving.** At the end, we saved the model. If the above is a model that I really want to keep, I can download it here from Collab or even save it to my google drive. Once saved, the model can be reloaded. Have a look at introduction_b.ipynb to see an instance of this model being reloaded and used.

## PyTorch Variant

And critically, we can even take advantage of Gemini and Copilot to do things even faster. Here is a Pytorch version of the same code. Note that asking for code generation doesn't always result in the easiest piece of code!

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# Define a transform to normalize the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download and load the training and test data
train_dataset = datasets.MNIST('.', download=True, train=True, transform=transform)
test_dataset = datasets.MNIST('.', download=True, train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=True)

# Define the neural network model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28) # Flatten the input
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        # Removed softmax activation here as CrossEntropyLoss expects raw logits
        x = self.fc3(x)
        return x

# Instantiate the model
model = Net()

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Start the training process
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

# Evaluate the model performance with test data
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Run the training and testing
epochs = 5
for epoch in range(1, epochs + 1):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)

# Save the model
torch.save(model.state_dict(), "mnist_model.pth")

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.302284
Train Epoch: 1 [6400/60000 (11%)]	Loss: 1.206160
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.531754
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.569354
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.316378
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.437994
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.281075
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.197875
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.106048
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.093642

Test set: Average loss: 0.0002, Accuracy: 9396/10000 (94%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.171442
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.213256
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.148007
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.201412
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.094927
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.212129
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.213358
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.147114
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.061058
T

The code above is training a Deep Neural Network based model to classify handwritten images like the ones below to digits. It is using state of the art optimisers, normalising the data, and gets 97% accuracy in just 5 epochs of training. It is taking advantage of the Keras interface to Tensorflow, and subject to getting your training data organised in the right way, it can be easily adopted to new problems.

<img width="200" src="https://drive.google.com/uc?id=1vNVU20xo-QDsySYSg7mCal3QTgvaksaK"/>

Therefore we can say that it is very easy to take some code that we find online to get a basic Deep Learning model training and producing results. This is great, and the work that the developers of Tensorflow and libraries like it have done to allow users to rapidly prototype models is simply amazing.

That said, if all we do is copy and paste code -- even with alterations -- we seldom learn and find it very difficult to apply learnings to novel or more challenging situations. For this reason, this coarse is not simply a guide to copying and pasting -- you don't need help to do that -- instead this course tries to give you a good understanding of some of the most important concepts that underpin the Deep Learning approach. Putting it simply, I want to  give you an understanding of why things are the way that they are.

Of course theory is useless without practice, so we will keep theory and practice interleaved for you.

### Content Covered

In this Deep Learning course we will trace through the most important concepts in Deep Learning such as the Feedforward Neural Networks, Convolutional Neural Networks, Recurrent Neural Networks and Transformers.

The course will place emphasis on introducing concepts that are commonly used in practical Deep Learning research and application. An emphasis on running examples will be made. As such a lot of lecture notes will be provided in the style of Jupyter notebooks that a student can download, edit and run on their own machines. I will also provide lecture slides on a week by week basis, but the Jupyther notebooks are the main source of information.

Some content commonly found in other courses will not be covered here. For example the following will not be addressed:

 * The history of neural networks
 * The biological inspiration of neural networks
 * Unsupervised learning of neural networks
 * Ethical Issues and Explainability
 * Details on the Backpropogation algorithm
 * Details on the gradient descent algorithm
 * Reinforcement Learning

Many of the issues above will be covered by the Advanced topics in Deep Learning class which runs alongside this module.

Speaking of videos, this course is not intended to be self-contained. While detailed notes with working examples will be given from week to week as Jupyter notebooks, links to external resources such as videos from Geoff Hinton and Andrew Ng will also be suggested for additional coverage or for a different perspective on a given question.

## Prerequisites / corequisites

This course is ideally intended as an Introduction to Deep Learning for students who have already completed undergraduate or Masters level modules on:

 * Artificial Intelligence, and, or
 * Machine Learning

A Machine Learning course is a co-requisite for this course. In other words, while it would be ideal that the Machine Learning module be taken already, it is also acceptable if you are taking Machine Learning in parallel to this module. Students have taken this module without a general 'Machine Learning' class, but you will have more to do in terms of digging into background ideas.

The module is suitable for someone who has already taken an online module in Deep Learning or otherwise consulted books etc. While no new material will be covered, the course will give you an opportunity to reflect on important topics with the support of fellow students.

All assignments must be coded up using Python with related libraries. If you have not already coded in Python, now is a good time to start. Introductory tutorials to Python and the use of the SKLearn (scikit) package should be consulted. In short, make sure that you can load a data set, perform k-fold cross validation of an SVM based classifier, and then run and present metrics such as the F1 score.


### Learning Outcomes

Our Learning Outcomes cover the following:

1. Understand the common elements and architectural constructions used in Deep Learning
2. Apply and critique Deep Learning Architectures to typical deployments in areas such as image processing$
3. Examine the challenge of representations and data types as used in the context of Deep Learning.
4. Examine the challenge of overfitting and data bias in the context of Deep Learning solution development.
5. Describe and critique common techniques for deploying and training deep neural networks.

Knowing what these are in advance isn't particularly useful, but some students do ask for them. Also it is worth noting that we will go beyond these simple few learning outcomes to give you as rounded as possible a view on particular Deep Learning based modelling.

## Course Structure

This is a 13 class programme. The course structure is subject to change and this content should be considered indicative. The earlier weeks focus in detail on standard feed forward neural networks and avoiding overfitting. As we move through the course we will move to other architectural variations.

* Week 1 - Introduction
* Week 2 - Neural Network Essentials
* Week 3 - Training and Loss Functions
* Week 4 - Better Activations
* Week 5 - Preventing Overfitting
* Week 6 - Convolutional Neural Networks
* Week 7 - Review Week (Bank Holiday)   
* Week 8 - Text and Representations     
* Week 9 - Recurrent Neural Networks    
* Week 10 - Transformers
* Week 11 - Hardware for Training and Deployment
* Week 12 - Transfer Learning & Fine Tuning
* Week 13 - ML Ops and Project Management

### Delivery Model

The course is a 5 ECTS course at Level 9/10. There are three contact hours per week, but students are expected to engage in a significant amount of self-study each week. The contact hours will be used for content delivery, group discussion, tests, reviewing assignments, and addressing any issues with respect to the delivery of the course. With the exception of Week 1, I will try to make lecture notes available in advance of the class. Therefore, students are strongly encouraged to study the content for each module in advance of that class. All content will be made available in the Applied Deep Learning module on Brightspace.

## Assignments and Assessment
This is a 50\% Exam and 50\% Continuous Assessment course. The assessment of the course will be broken down as follows:

The exam may use a combination of short and long questions and will aim to test your knowledge of the concepts introduced in the course.

The project meanwhile is partly a coding / modelling task which is designed for a post LLM world. Hence the focus is not on basic coding tasks, but instead on understanding correct design and methodology. Specific instructions on the project will be provided midway in the course. The project will be due at the end of the semester. All students are required to orally present on their project and answer questions on their models / code at the end of the year. The project has an optional group element.

## Coding
In this course we will use Python extensively for all examples and assignments. Rather than using vanilla Python we will where appropriate make use of Python packages that provide enhanced functionality for numerical computing and Deep Learning. Examples have been coded up using Python 3.11. You should set up your environment for Python 3.11 for minimal problems replicating code and examples.

If you aren't familiar with either Python, now is the time to get familiar. We will use tutorail time in week 1 to discuss these and work through installations.

Course notes and all examples will be coded up in the **Jupyter Notebook** environment. Notebooks will be made available on Brightspace for download. Students who do not already use Jupyter Notebook should install Jupyter Notebook. Note however that most notebooks will also be tested and deployed on Google Colab. Again, this will be discussed in the Week 1 tutorial.

For your work environment, you should make use of a virtual environment to manage your python code. I suggest that you setup your environment using venv and pip. I discourage the use of Anaconda as it takes away too much control from you. For coding, Jupyter is good for short examples. But for longer projects, there are many good IDEs available including Atom (with suitable plugins) and VS Code.


### Hardware Required

Students are expected to have running installations of scipy and TensorFlow running on their own machines for testing. Your machine does not have to be particularly fast for most problems. For the first half of the course in particular, you should be able to run sample code on a moderate machine. A GPU is not needed for that code. However for some models discussed in the 2nd part of the course, more serious hardware would be required.

Google Codelab is a nice platform where you can run even big models for free -- though you are limited in the amount of time the model can be left running for.


# Appendices

Below you will find a couple of useful reference posts for issues such as notation and linear algebra in python.

## Appendix A - Notation

Given a training set we talk about:
 * $m$ = number of training examples
 * $x$'s = input variables or features
 * $y$'s = output variable or target
 * $(x,y)$ = one training example
 * $(x^{i},y^{i})$ = refers specifically to the ith training case

## Appendix B - Linear Algebra in Python

In python we can use the `numpy` library to easily define and perform operations on vectors and matrices.

### Matrices
The term **matrix** refers to a 2D rectangular array of numbers.

In [5]:
import numpy as np

# We can define a matrix directly from a string of numbers where rows are delimited by semi-colons
A = np.matrix('1 2 3; 4 5 6')
print(A)

# Alternatively we can define the same matrix from a series of vectors where each vector defines a row of the matrix
B = np.matrix([[1, 2, 3], [4, 5, 6]])
print(B)

# Alternatively we can use numpy's array constructor to creaea a 2D array, i.e., our matrix
B = np.array([[1, 2, 3], [4, 5, 6]])
print(B)

[[1 2 3]
 [4 5 6]]
[[1 2 3]
 [4 5 6]]
[[1 2 3]
 [4 5 6]]


A matrix will have a certain *dimensionality* defined in terms of the number of rows and the number of columns. We can use standard number theory notation to define the dimensionality of the matrix. For example we can define the set of matrices of real numbers with 3 columns and 2 rows as $R^{3x2}$.

The **shape** function in numpy will return the number of rows and columns for a given matrix.

In [6]:
print(A.shape)

(2, 3)


We refer to individual objects within the matrix as **elements**. We use subscript notation on the matrix name in order to refer to individual objects. In general $M_{i,j}$ will refer to the element found on the $i^{th}$ row and $j^{th}$ column of M.

Numpy supports a wide range of methods for indexing and slicing arrays which we will not cover here. In the simple case we can however use indices to operate directly on the matrix as follows.

In [7]:
print(A[1,1])

5


Note that indexing on the rows and columns of numpy matrices begins at 0.

### Vectors
A vector is a 1D array and as such can be thought of as a special case of a matrix with only one column.

While the vector is in general a special case of a matrix, we typically use different operators to create and work with vectors.

In [8]:
# create a numpy vector by way of a stadard python list fed into the array constructor
v1 = np.array([2,3,1,0])
print(v1)

# note that this is not equivilent to attempting to create the array as a single row of a matrix
v2 = np.matrix([[2,3,1,0]])
print(v2)

# index an element in the vector
print(v1[1])

[2 3 1 0]
[[2 3 1 0]]
3


### Matrix and Vector Basic Operations

We can add and subtract matrices which are of the same dimensionality to result in a new matrix which is of that same dimensionality.

In [9]:
C = A + B
print(C)
D = C - A
print(D)

[[ 2  4  6]
 [ 8 10 12]]
[[1 2 3]
 [4 5 6]]


We can do the same for vectors

In [10]:
v2 = np.array([10,20,30,40])

v3 = v1 + v2
print(v3)
v3 = v2 - v1
print(v3)

[12 23 31 40]
[ 8 17 29 40]


We can also directly apply scalar multiplication and division operations to matrices and vectors.  

In [11]:
E = A * 2
print(E)
F = A / 2
print(F)
v4 = v2 * 2
print(v4)
v5 = v2 / 2
print(v5)

[[ 2  4  6]
 [ 8 10 12]]
[[0.5 1.  1.5]
 [2.  2.5 3. ]]
[20 40 60 80]
[ 5. 10. 15. 20.]


### Inner / Scalar Product
The Inner Product or Scalar Product of two vectors is the scalar result of summing the pairwise products of elements in two vectors of equal length. In geometric space the Scalar Product is often interpreted as a distance metric between two points in that space. This is often used for example to calculate document similarities and in clustering.

In python we can calculate the scalar product of two vectors using numpy's inner function.

In [12]:
v3 = np.dot(v1,v2)
print(v3)

110


Note that the dot function will also produce this result when applied to two vectors. However the dot function can also be applied to matrices and higher-order arrays.

### Matrix-Matrix Multiplication
Given two matrices we can calculate the cross product of these matrices so long as the number of rows in the first matrix equals the number of columns in the second.

For two matrices A and B, the dimensionality of the resultant matrix product is given by:

\begin{equation}
R_{A}C_{A} \times R_{B}C_{B} = R_{A}C_{B}
\end{equation}

The operation for calculating the matrix product is straightforward: the entry for row i column j in the resultant matrix C is the dot product of the i$^{th}$ row of A and the j$^{th}$ column of B.

![matrix matrix multiplication](figures/img792.gif)

In python we can use the numpy function **matmul** to perform matrix-matrix multiplication.

In [13]:
G = np.matrix('1 2; 3, 4; 5, 6')
print(G)
H = np.matmul(A,G)
print(H)

[[1 2]
 [3 4]
 [5 6]]
[[22 28]
 [49 64]]


Remember that matrix matrix multiplication is not commutative.
\begin{equation}
  A \times B \neq B \times A
\end{equation}

In [14]:
A = np.matrix('1, 2, 3; 4, 5, 6; 7, 8, 9')
B = np.matrix('1, 1, 1; 2, 2, 2; 3, 3, 3')
print(A)
print(B)
print(np.matmul(A,B))
print(np.matmul(B,A))

[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[1 1 1]
 [2 2 2]
 [3 3 3]]
[[14 14 14]
 [32 32 32]
 [50 50 50]]
[[12 15 18]
 [24 30 36]
 [36 45 54]]


### Matrix Identity and Inverse

We can define the identity $I$ of a matrix which in general allows the commutation property to hold.

\begin{equation}
 A \times I = I \times A = A
\end{equation}

Here $I$ is the Identity Matrix which is a square matrix where all diagonal elements are = 1 and all non-diagonal elements are = 0.

Numpy allows us to easily define an identity matrix with a specified number of rows and columns.

In [15]:
I = np.identity(3)
print(I)
print(A)
print(np.matmul(A,I))
print(np.matmul(I,A))

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]


Just as we can define the inverse for a real number $X \in R$ as $\frac{1}{X}$, we can also define the inverse for a matrix.

Beginning first with the case of real numbers, we note that:

\begin{equation}
 X \times INV(X) = I
\end{equation}

where I is the identity for real numbers - which is 1.

This gives us an intuition of how the inverse is defined for matrices:  

\begin{equation}
 A \times INV(A) = I
\end{equation}

i.e., the inverse of a matrix A should be defined such that the matrix product of A by it produces an identity matrix.

While we straightforwardly calculate the inverse of a real number X as $\frac{1}{X}$, the calculation of the inverse of a matrix is more complicated and involves calculations of Determinants and Cofactors of matrices which we will not consider here. Fortunately we can of course calculate the inverse directly in numpy.

In [16]:
import numpy.linalg as la
X = np.matrix('100, -200, 403; 44, -5, 607; 27, 98, -59')
B = la.inv(X)
print(B)
print(np.matmul(X,B))

# note that the process doesn't work very well when the candidate matrix is close to 0
print("Uncomment lines in the block to run this - but expect an error!")
# B = la.inv(A)
# print(np.matmul(A,B))

[[ 0.00746988 -0.00349497  0.01506633]
 [-0.0023959   0.00211775  0.00542254]
 [-0.00056121  0.00191823 -0.00104746]]
[[ 1.00000000e+00  4.31512465e-17  2.29850861e-17]
 [-1.07336015e-17  1.00000000e+00 -5.16080234e-17]
 [ 1.55040911e-17 -3.10081821e-17  1.00000000e+00]]
Uncomment lines in the block to run this - but expect an error!


### Matrix Transpose
We can also define the transpose of a matrix $A^{T}$ as a matrix such that rows and columns of $A$ are reversed.

\begin{equation}
   A_{ij} = A_{ji}
\end{equation}

In [17]:
C = A.T
print(A)
print(C)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[1 4 7]
 [2 5 8]
 [3 6 9]]


### Do I need all this Linear Algebra?

Knowledge of the issues above is typically not needed just to run basic Deep learning modeling tasks. However, in the first number of weeks while we are figuring out why things work they way that they do, this is essential.